<center>
    <img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="900">
</center>  

# Yet another math for ML course

# Домашнее задание 2 (продвинутая группа): SVD в ML

**Социальный бонус [1 балл]:** у нас онлайн-магистратура, но это не отменяет движа. Сходите куда-нибудь с однокурсниками и прикрепите фото с этого похода сюда.

## Формат проверки работы: AI-агент + ассистент

Что это значит для вас:

* тетрадка должна выполняться сверху вниз без ошибок — перед сдачей сделайте `Restart & Run All`;
* ответы на вопросы пишите в отведённые ячейки после «Ответ:», коротко и по существу. Ответы — часть задания и оцениваются наравне с кодом: работающий код с пустым «Ответ:» полного балла не получает;
* у графиков должны быть подписаны оси и заголовки — агент читает картинки, но не умеет угадывать, что на них;
* комментарии проверяющего придут вместе с оценкой. Если вы с ней не согласны — напишите ассистенту, указав конкретное задание и что именно, по вашему мнению, проверено неверно;
* последнее слово в выставлении оценки будет за ассистентом. 

> В какие-то из работ ассистенты придут сами для перепроверки баллов.

Пользоваться ChatGPT, Claude и прочими как справочником и напарником можно.

Сдавать сгенерированное целиком решение, которое вы не понимаете, — нельзя. Вы обязаны разобраться в каждой строчке своего кода: часть студентов будет вызвана на защиту домашки, и если защита провалится, оценка за работу обнулится.

In [ ]:
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import convolve2d
from skimage import data, img_as_float

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

np.random.seed(42)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 12

В базовой домашке мы с помощью SVD исследовали картинку и сжимали её. В этой домашке мы посмотрим на ту же самую идею, но в разрезе трёх ML-ных кейсов.

Главная идея домашки:

> Низкий ранг — это всегда экономия. Вопрос только в том, что именно мы экономим.

Дальше мы будем всё время смотреть на сингулярные числа и делать по ним выводы. Для начала давайте потренируемся делать это на матрицах, про которые мы всё знаем заранее.

Возьмём четыре матрицы $100 \times 100$ и посмотрим на их спектры: точную матрицу ранга 1, точную матрицу ранга 3, матрицу ранга 3 с добавленным шумом и чистый шум.

In [ ]:
def random_matrix(rank, n=100):
    M = sum(np.outer(np.random.randn(n), np.random.randn(n)) for _ in range(rank))
    return M / np.linalg.norm(M)

In [ ]:
rank_1 = random_matrix(1)
rank_3 = random_matrix(3)

noise = np.random.randn(100, 100)
noise = noise / np.linalg.norm(noise)

examples = {
    "ранг 1": rank_1,
    "ранг 3": rank_3,
    "ранг 3 + шум": rank_3 + 0.3 * noise,
    "чистый шум": noise,
}

In [ ]:
for name, M in examples.items():
    s = np.linalg.svd(M, compute_uv=False)
    plt.plot(s[:20] / s[0], marker="o", label=name)

plt.xlabel("номер сингулярного числа")
plt.ylabel("сингулярное число")
plt.legend()
plt.show()

Что мы видим?

- У точных матриц ранга 1 и ранга 3 спектр ожидаемо обрывается после ранга и дальше идут нули. Ровно то, что обещает теория.

- Как только мы подмешиваем шум, формальный ранг мгновенно становится полным. При этом, на картинке мы по-прежнему видим три больших числа и плато из всякой мелочи после него.
  
- У чистого шума спектр ползёт вниз медленно.

Получается, что сам по себе ранг матрицы почти бесполезен, а вот форма спектра говорит о данных всё. Всё, что идёт после резкого обрыва - мусор.

## Задача 1 (ты неси меня река)

В этом задании вам предстоит работать с подмножеством датасета, повествующего о химическом составе воды в некоторых реках. Данные лежат в файле `dataset_final`.

Данные взяты из European Environment Agency Waterbase Rivers.

Файл `dataset_final` содержит три объекта:

```python
index_list, feature_list, data_matrix
```

* `index_list` — идентификаторы станций наблюдения;
* `feature_list` — признаки вида `'2005 Nitrate'`, то есть пара «год + показатель»;
* `data_matrix` — сама матрица $X \in \mathbb{R}^{500 \times 532}$.

Мы хотим применить SVD и получить новые координаты:

$$
X \approx U_k \Sigma_k V_k^T .
$$

Строки матрицы $U_k \Sigma_k$ — это координаты рек в скрытом пространстве. Если первые две координаты действительно ловят структуру, двумерная картинка станет осмысленной: похожие реки окажутся рядом.

Но есть нюанс. Реальные табличные данные почти никогда нельзя просто взять и засунуть в SVD.

__(0.5 балла) Задание 1:__ загрузите данные и посмотрите, что внутри.

Выведите форму матрицы, минимум, максимум и долю нулей.

In [ ]:
# YOUR CODE HERE
# ฅ^•ﻌ•^ฅ

with open("data/dataset_final", "rb") as f:
    index_list, feature_list, data_matrix = pickle.load(f)

X_raw = ...

А теперь сделайте то, чего делать не стоит: примените SVD прямо к сырой матрице. Выясните, какая доля $\sigma_1^2 / \sum_i \sigma_i^2$ приходится на первую компоненту.

In [ ]:
# YOUR CODE HERE
# ฅ^•ﻌ•^ฅ

Посмотрите у каких признаков оказались самые большие по модулю коэффициенты в `Vt[0]`.

In [ ]:
# YOUR CODE HERE
# ฅ^•ﻌ•^ฅ

**Вопрос:** какую долю дисперсии забрала первая компонента и что она, судя по `Vt[0]`, нашла в данных? Можно ли назвать это структурой?

**Ответ:**

...

Итак, с сырой таблицей получилась глупость. Давайте её чинить:

- **Пропуски.** В этой матрице их нет в виде `NaN` — вместо них стоят нули. Это опаснее: ноль это законное значение показателя, и SVD не отличит «кислорода не измеряли» от «кислорода не было».

- **Выбросы.** Кроме азота за 2008 год, в данных есть `pH = 777` (шкала pH заканчивается на 14). Это явно опечатка, которая выглядит как выброс. Такие значения перетянут на себя первые компоненты.

- **Масштабы.** Ортофосфаты измеряются десятыми долями, насыщение кислородом — сотнями. Без приведения к общему масштабу SVD будет смотреть на самый крупный по числам показатель, а не на самый содержательный.

__(0.5 балла) Задание 2:__ приведите матрицу в порядок.

Разумный минимум такой: превратить нули в честные `np.nan`, обрезать выбросы сверху по 99-му процентилю, выкинуть безнадёжно пустые столбцы и строки, заполнить оставшиеся дыры медианой столбца и привести признаки к общему масштабу.

Обрезать выбросы нужно внутри каждого **показателя**, а не по всей таблице сразу: у pH и у фосфатов совершенно разные разумные диапазоны. Название показателя достаётся из признака так: `" ".join(feature.split()[1:])`.

Пороги фильтрации выбирайте сами.

In [ ]:
# YOUR CODE HERE
# (⌐■_■)

X = np.where(X_raw == 0, np.nan, X_raw)
indicator_of = [" ".join(f.split()[1:]) for f in feature_list]

X_prep = ...
features = ...
stations = ...

**Коротко опишите свою предобработку и объясните, зачем нужен каждый шаг:**

...

__(0.5 балла) Задание 3:__ посчитайте SVD предобработанной матрицы и посмотрите на её спектр — как в разминке, только теперь на настоящих данных.

Нарисуйте рядом сами сингулярные числа и накопленную долю энергии

$$
\frac{\sigma_1^2 + \ldots + \sigma_k^2}{\sigma_1^2 + \ldots + \sigma_r^2},
$$

а на второй график нанесите горизонтальные линии на уровнях 50%, 80% и 90% — так сразу видно, сколько компонент нужно.

In [ ]:
# YOUR CODE HERE
# (⌐■_■)

**Вопрос:** есть ли у этих данных низкоранговая структура? Сколько компонент нужно, чтобы объяснить 80% энергии, и как это соотносится с числом признаков? Сравните с сырой матрицей из задания 1 — там первая компонента забирала почти всю энергию, но структурой это не было. В чём разница?

**Ответ:**

...

__(0.5 балла) Задание 4:__ прочитайте, что означают первые компоненты.

Строки `Vt[0]`, `Vt[1]` и `Vt[2]` показывают, из каких исходных признаков собрана каждая скрытая координата: чем больше коэффициент по модулю, тем сильнее признак на неё влияет.

Выведите для каждой из трёх первых компонент топ-8 признаков вместе со знаками коэффициентов и попробуйте назвать компоненту словами.

Знак компоненты сам по себе ничего не значит: умножьте её на $-1$, и ничего не изменится. Смысл имеет только противопоставление плюсов и минусов внутри одной компоненты.

In [ ]:
# YOUR CODE HERE
# (☉_☉)

for c in range(3):
    top = ...

**Интерпретация компонент:**

* Компонента 1: ...
* Компонента 2: ...
* Компонента 3: ...

## Задача 2 (а орешки не простые)

Свёртка — главный рабочий инструмент обработки изображений. Фильтр размытия усредняет соседей, фильтр границ подсвечивает резкие переходы. Ну а в свёрточных нейросетях обучаются сотни фильтров, детектирующих различные паттерны. Когда учишь кучу свёрток, хочется экономить в параметрах и скорости вычислений.

Если ядро свёртки имеет малый ранг, свёртку можно посчитать сильно быстрее. Посмотрим, почему. Пусть ядро имеет ранг 1, то есть раскладывается в произведение столбца на строку:

$$
K =
\frac{1}{9}
\begin{pmatrix}
1 & 1 & 1\\
1 & 1 & 1\\
1 & 1 & 1
\end{pmatrix}
=
\frac{1}{3}
\begin{pmatrix}
1\\1\\1
\end{pmatrix}
\cdot
\frac{1}{3}
\begin{pmatrix}
1 & 1 & 1
\end{pmatrix}
= c \, r^T .
$$

Такие свёртки, разваливающиеся в произведение двух векторов, называют **сепарабельными**. Они позволяют сильно сэкономить.

Пусть $A$ - наше изображение, а $*$ применение свёртки. Свёртка ассоциативна, поэтому

$$
A * K = A * (c \, r^T) = (A * c) * r^T .
$$

Слева — одна двумерная свёртка ядром $k \times k$, справа — две одномерные: сначала по вертикали, потом по горизонтали. Считая операции на один пиксель, слева получаем $k^2$ умножений, справа $2k$: для $k = 51$ это 2601 против 102.

Вспомним, что SVD позволяет переписать исходную матрицу в виде суммы

$$
K \approx \sigma_1 u_1 v_1^T + \ldots + \sigma_r u_r v_r^T .
$$

Получается, если взять произвольную свёртку, сделать её SVD и оставить только первую компоненту, то мы получим её сепарабельное приближение.

__(0.5 балла) Задание 5:__ разомнёмся на фильтре границ.

Свернём картинку ядром

$$
B =
\begin{pmatrix}
-1 & -1 & -1\\
-1 & 8 & -1\\
-1 & -1 & -1
\end{pmatrix}.
$$

In [ ]:
img = img_as_float(data.camera())

B = np.array([[-1, -1, -1],
              [-1,  8, -1],
              [-1, -1, -1]], dtype=float)

edges = convolve2d(img, B, mode="same", boundary="symm")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, pic, title in zip(axes,
                          [img, edges, np.abs(edges)],
                          ["исходная картинка", "результат свёртки", "модуль результата"]):
    ax.imshow(pic, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

Посмотрите на две вещи: чему равна сумма элементов ядра и какие у него сингулярные числа.


In [ ]:
# YOUR CODE HERE
# ( ͡⊙ ͜ʖ ͡⊙)

**Вопрос:** что этот фильтр делает с изображением и почему сумма его элементов равна нулю? И что удивительного в его ранге?

**Ответ:**

...

__(1.5 балла) Задание 6:__ проверим, что сепарабельность действительно ускоряет.

Возьмите фильтр усреднения $K = \frac{1}{k^2}\mathbf{1}\mathbf{1}^T$ и примените его к картинке двумя способами: одной двумерной свёрткой и двумя одномерными — сначала столбцом формы `(k, 1)`, потом строкой формы `(1, k)`.

Проделайте это для $k \in \{3, 7, 15, 31, 51\}$, каждый раз убеждаясь через `np.allclose`, что результаты совпали, и замеряя время обоих способов через `time.perf_counter`.

In [ ]:
# YOUR CODE HERE
# ( ͡⊙ ͜ʖ ͡⊙)

Нарисуйте, как растёт время у обоих способов и как меняется ускорение с ростом размера фильтра.

In [ ]:
# YOUR CODE HERE
# ( ͡⊙ ͜ʖ ͡⊙)

**Вопрос:** как ускорение зависит от размера фильтра и почему именно так? Совпадает ли оно с тем, что обещает подсчёт операций?

**Ответ:**

...

__(1 балл) Задание 7:__ а теперь сожмём сам фильтр.

Фильтр границ $B$ имеет ранг 2, сепарабельным он не является. Но никто не мешает построить его лучшее приближение ранга 1 через SVD: $B_1 = \sigma_1 u_1 v_1^T$.

Посчитайте SVD матрицы $B$, соберите $B_1$, выведите его и посмотрите на относительную ошибку ядра $\|B - B_1\|_F / \|B\|_F$. Заодно посмотрите, чему теперь равна сумма элементов.

In [ ]:
# YOUR CODE HERE
# (っ˘ڡ˘ς)

Ошибка ядра получилась умеренная. Прежде чем считать дальше, попробуйте угадать: ошибка самого результата свёртки выйдет примерно такой же, больше или меньше?

Теперь примените оба фильтра к картинке, нарисуйте результаты рядом и посчитайте относительную ошибку результата.

In [ ]:
# YOUR CODE HERE
# (っ˘ڡ˘ς)

**Вопрос:** насколько разошлись две ошибки и почему так вышло? Что это говорит про теорему Эккарта — Янга?

**Ответ:**

...

__(1 балл) Задание 8:__ то же самое, но на большом ядре.

Когда объектив промахивается мимо фокуса, каждая светящаяся точка сцены расплывается на матрице в кружок — так получается боке. Расфокус — это свёртка с диском: внутри круга единицы, снаружи нули, всё поделено на сумму. Чем сильнее промах, тем больше радиус, а цена свёртки растёт как $k^2$, поэтому в графике и на телефонах такое размытие всегда считают приближённо.

Сепарабельным диск заведомо не является: круг невозможно получить произведением столбца на строку. Но посмотрим на его спектр.

Постройте диск радиуса 15 внутри матрицы $31 \times 31$, нормируйте на сумму элементов и выведите ядро рядом со спектром. Заодно посчитайте формальный ранг — он окажется неожиданным.

In [ ]:
# YOUR CODE HERE
# ฅ^•ﻌ•^ฅ

Прежде чем считать дальше, вспомните прошлое задание: там ошибка ядра 26% обернулась ошибкой результата 995%. Попробуйте угадать, что будет здесь.

Теперь для рангов $r \in \{1, 2, 3, 5\}$ соберите приближение $K_r$ и посчитайте **обе** ошибки — ядра и результата свёртки, — а заодно время: приближение ранга $r$ считается как сумма $r$ сепарабельных свёрток.

Разложить одно слагаемое на столбец и строку помогает такой трюк: $\sigma_i u_i v_i^T = (\sqrt{\sigma_i}\, u_i)(\sqrt{\sigma_i}\, v_i)^T$.

В конце нарисуйте рядом полное размытие, приближение ранга 1 и их разницу.

In [ ]:
# YOUR CODE HERE
# ฅ^•ﻌ•^ฅ

**Вопрос:** какой ранг вы бы выбрали для этого ядра и почему? В задании 7 ошибка ядра 26% превратилась в 995% на результате, здесь ошибка ядра того же порядка — а что вышло с результатом? Из-за чего такая разница?

**Ответ:**

...

## Задача 3 (кто убил Лору Палмер)


LoRA расшифровывается как **Low-Rank Adaptation**, и именно так сегодня дообучают большие языковые модели на одной видеокарте.

Идея простая. Пусть в сети есть линейный слой с матрицей весов $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$. Обычное дообучение меняет всю матрицу:

$$
W_{\text{new}} = W_0 + \Delta W .
$$

Это дорого: нужно хранить и обучать $d_{out} \cdot d_{in}$ чисел, а таких слоёв в модели сотни.

LoRA предлагает обучать не всю матрицу обновления, а её низкоранговое представление:

$$
\Delta W \approx A B, \qquad A \in \mathbb{R}^{d_{out} \times r}, \quad B \in \mathbb{R}^{r \times d_{in}} .
$$

Параметров становится $r(d_{out} + d_{in})$ вместо $d_{out} d_{in}$.

Но работает это только при одном условии: **обновление весов действительно должно быть близко к низкоранговому**. Иначе мы просто выбросим часть выученного.

__(0.5 балла) Задание 9:__ посчитайте, сколько параметров экономит LoRA.

Пусть $d_{in} = d_{out} = 4096$ — типичный размер слоя в языковой модели. Сравните число параметров в полном обновлении $\Delta W$ и в LoRA-обновлении ранга $r$ для $r \in \{1, 2, 4, 8, 16, 32, 64\}$, а потом нарисуйте, какую долю от полного обновления занимает LoRA.

In [ ]:
# YOUR CODE HERE

### ╰( ͡° ͜ʖ ͡° )つ▬▬ι═══════  bzzzzzzzzzz
# may the rank be with you

**Вопрос:** какого порядка получается экономия и как она зависит от размера слоя?

**Ответ:**

...

__(1 балл) Задание 10:__ проверим, от чего зависит успех LoRA.

Соберите две матрицы обновления размера $256 \times 256$ с одинаковой нормой, но разными спектрами: у первой сингулярные числа быстро убывают (например, $\sigma_i = e^{-i/5}$), у второй почти не убывают (например, все равны единице). Это та же пара сюжетов, что «структура» и «шум» из разминки, только теперь спектр задаём мы сами.

Матрица с нужным спектром собирается как $U \Sigma V^T$, где $U$ и $V$ — случайные ортогональные матрицы. Такую матрицу удобно получить QR-разложением случайной гауссовской.

In [ ]:
# YOUR CODE HERE

### ╰( ͡° ͜ʖ ͡° )つ▬▬ι═══════  bzzzzzzzzzz
# singular but not lonely

Теперь для каждой матрицы постройте лучшие приближения разных рангов через SVD и посмотрите, как убывает относительная ошибка

$$
\frac{\|\Delta W - \Delta W_r\|_F}{\|\Delta W\|_F}.
$$

Спектры и кривые ошибок удобно нарисовать рядом.

In [ ]:
# YOUR CODE HERE

### ╰( ͡° ͜ʖ ͡° )つ▬▬ι═══════  bzzzzzzzzzz
# singular but not lonely

**Вопрос:** в каком случае LoRA сработает, а в каком нет? Сформулируйте условие её применимости.

**Ответ:**

...

До сих пор мы работали с придуманными матрицами. Но LoRA — это утверждение о реальности: что настоящее обновление весов при дообучении оказывается почти низкоранговым. Проверим на живой сети.

Сеть возьмём крошечную, чтобы всё считалось за секунды: один скрытый слой на 256 нейронов, датасет `digits` из `sklearn` — картинки цифр $8 \times 8$.

Сценарий такой. Сначала учим сеть распознавать цифры как есть. Потом меняем задачу: те же цифры, но сдвинутые на один пиксель по диагонали. Для сети это заметный сдвиг домена, точность падает почти до случайной. Дообучаем её на новых данных и смотрим, чем стала отличаться матрица весов.

__(1 балл) Задание 11:__ получите настоящее обновление весов.

Обучите `MLPClassifier(hidden_layer_sizes=(256,))` на исходных цифрах и сохраните матрицу первого слоя: `W0 = clf.coefs_[0].copy()`. Проверьте, что на сдвинутых цифрах точность заметно падает, а потом дообучите ту же сеть на них через `warm_start=True` и сохраните новую матрицу `W1`.

Сдвиг картинок делается так:

```python
def shift_images(X, dx=1, dy=1):
    imgs = X.reshape(-1, 8, 8)
    return np.roll(np.roll(imgs, dy, axis=1), dx, axis=2).reshape(-1, 64)
```

Обязательно убедитесь, что после дообучения точность выросла: если оптимизация не сошлась, то $\Delta W$ будет шумом, и все дальнейшие выводы окажутся выводами про шум.

In [ ]:
# YOUR CODE HERE

### ╰( ͡° ͜ʖ ͡° )つ▬▬ι═══════  bzzzzzzzzzz
# may the rank be with you

**Вопрос:** насколько упала точность от сдвига и восстановилась ли она после дообучения? Почему без этой проверки нельзя смотреть на спектр $\Delta W$?

**Ответ:**

...

__(1.5 балла) Задание 12:__ сравните спектр обновления со спектром самих весов.

Постройте накопленную долю энергии для $\Delta W$ и для $W_0$ **на одной картинке**: сравнивать надо именно с базой, иначе непонятно, много это или мало. Заодно посчитайте относительную ошибку rank-r приближения обновления для нескольких рангов.

Главный вопрос: **обновление более низкоранговое, чем сами веса, или нет?**

In [ ]:
# YOUR CODE HERE

### ╰( ͡° ͜ʖ ͡° )つ▬▬ι═══════  bzzzzzzzzzz
# singular but not lonely

**Вопрос:** сработала бы LoRA для этого слоя? И насколько далеко можно распространять полученный вывод?

**Ответ:**

...

---

### Что мы в итоге сэкономили

Домашка начиналась с тезиса: низкий ранг — это всегда экономия, вопрос только в том, что именно мы экономим. Теперь на него можно ответить.

- В реках мы экономили в числе фичей. По сути мы получили обычный PCA с выделением главных компонент.

- В свёртках мы экономили вычисления. Сепарабельное ядро превращает $k^2$ умножений на пиксель в $2k$, и на фильтре $51 \times 51$ это дало ускорение в десяток раз. А если ядро не сепарабельно, его можно приблизить суммой нескольких сепарабельных — и заплатить за скорость точностью.

- В LoRA мы экономили параметры.  Обновление весов слоя, вместо $d_{out} \cdot d_{in}$ чисел, хранится в $r(d_{out} + d_{in})$ — и работает это потому, что дообучение действительно меняет слой в немногих согласованных направлениях.

Но у экономии есть цена. SVD даёт лучшее приближение в норме Фробениуса. Она не знает, что для нас важно при решении задачи. У диска расфокуса ошибка ядра 32% стоила 1% на результате, у фильтра границ ошибка 26% стоила 995%.

> SVD помогает экономить, но делать это надо осторожно :)